# Guia científico comentado — TCCT v3.2.1

Este notebook é deliberadamente **explicativo**. Ele acompanha o pipeline executável, mas não processa dados nem treina modelos.

A organização segue a lógica didática do notebook de referência anexado — motivação antes do código, justificativa física/computacional e relação com o objetivo científico — porém atualiza os pontos que já foram corrigidos no projeto:

- uso exclusivo do par interferométrico 0506–0603;
- back-geocoding pela `.grille` 3D, não por vértices ROI/KDTree;
- dataset v6 com 10.500 patches;
- coerência normalizada;
- `k_z` P derivado dos produtos Ha com referência 0007 cancelada;
- validação espacial por grupos;
- treinamento em duas fases, com seleção final entre Fase A e Fase B;
- retomada por época e por combinação fold–seed.

## Parte 1 — Problema científico e desenho experimental

### 1.1 Objetivo

O TCCT estima altura do dossel em floresta tropical densa a partir de três fontes complementares:

1. Landsat-5, com informação espectral e de cobertura;
2. PolInSAR Banda L, com sensibilidade estrutural e interferométrica;
3. PolInSAR Banda P, com maior penetração e coerência geralmente mais alta.

O CHM LiDAR é o alvo de supervisão e não entra como modalidade de entrada.

### 1.2 Por que não basta um split aleatório

Patches próximos compartilham solo, estrutura florestal, condições de aquisição e contexto espacial. Uma separação aleatória pode colocar vizinhos quase idênticos em treino e validação, inflando as métricas. O protocolo principal, portanto, usa blocos espaciais e GroupKFold.

### 1.3 O que é evidência principal

O resultado principal não é o melhor fold isolado. É a distribuição em cinco folds e três seeds, separando:

- variação espacial entre folds;
- variação de inicialização entre seeds.

O holdout 70/20/10 é complementar e não substitui essa avaliação.

## Parte 2 — Preparação geométrica e altimétrica

### 2.1 Geometria SAR

Os SLCs estão em coordenadas radar. Para fundi-los com Landsat e LiDAR, cada posição cartográfica precisa ser invertida para linha e coluna fracionárias no SLC.

A `.grille` ONERA contém amostras de:

`linha, coluna, altitude, longitude, latitude`.

A inversão é realizada por superfícies altimétricas estruturadas. O algoritmo foi validado internamente por round-trip e externamente com os pontos ROI.

### 2.2 Referência vertical

A `.grille` usa altitude elipsoidal compatível com GRS80. O DTM LiDAR foi identificado como necessitando transformação NGG1977 → elipsoidal. A grade geoidal oficial é aplicada antes do back-geocoding.

### 2.3 Interpolação complexa

A interpolação bilinear é aplicada separadamente às partes real e imaginária. Não se interpola apenas magnitude, pois isso destruiria fase e comprometeria a interferometria.

## Parte 3 — Landsat, LiDAR e construção do dataset

### 3.1 Landsat-5

As cenas de 29/08/2009 e 14/09/2009 são combinadas por qualidade externa ao alvo. A seleção usa QA, saturação, NoData, fração de pixels claros e proximidade temporal.

As oito variáveis são seis bandas de reflectância, NDVI e EVI.

### 3.2 CHM

O CHM é reconstruído por `DSM − DTM` após alinhamento rigoroso. Valores inválidos e NoData são mascarados antes da extração.

### 3.3 Dataset v6

Cada patch registra:

- tensores SAR complexos L/P;
- coerência e qualidade;
- atributos Landsat;
- CHM e máscara;
- coordenadas cartográficas;
- grupo espacial;
- janela LiDAR;
- qualidade geométrica SAR.

O dataset atual contém 10.500 patches válidos. Patches antigos v4/v5 não devem ser reutilizados.

## Parte 4 — Coerência e física RVoG

### 4.1 Coerência observada

A coerência normalizada exige o produto cruzado mestre–escrava e as potências locais de ambos. O produto interferométrico sem normalização não é suficiente.

### 4.2 Número de onda vertical

Para a Banda <LOCAL_PATH>
`kz(0506,0603) = 2π/Ha(0007,0603) − 2π/Ha(0007,0506)`.

Como os dois Ha compartilham a referência 0007, essa referência se cancela. O TCCT continua usando apenas o par 0506–0603.

### 4.3 Calibração sem vazamento

O sinal de `k_z` e a atenuação efetiva são escolhidos/calibrados apenas no treino de cada fold. A validação verifica cobertura, mas não participa do ajuste físico.

## Parte 5 — Arquitetura TCCT

### 5.1 Encoders

Landsat usa encoder real. L e P usam representações complexas para preservar relações de fase e amplitude.

### 5.2 Coatenção/gating

A fusão aprende pesos modais adaptativos. Esses pesos podem ser analisados, mas devem ser descritos como gating/coatenção real do modelo. Normas de ativação não devem ser rotuladas como atenção.

### 5.3 Decoder

O decoder produz regressão densa de altura. A avaliação inclui métricas por pixel e por média de patch, pois elas respondem a escalas diferentes do problema.

## Parte 6 — Treinamento em duas fases

### 6.1 Fase A

O modelo minimiza apenas a perda supervisionada composta por termos de pixel, patch, gradiente e concordância.

### 6.2 Fase B

A Fase B começa do melhor checkpoint da Fase A e adiciona gradualmente a regularização RVoG. Como a função objetivo muda, o RMSE de validação pode piorar mesmo partindo do melhor modelo supervisionado.

### 6.3 Seleção final

O pipeline compara o melhor checkpoint das duas fases. Se a Fase B não melhorar o RMSE de patch, a Fase A permanece como modelo final. Assim, a física é testada sem forçar a substituição por um modelo pior.

## Parte 7 — Retomada computacional

### 7.1 Nível combinação

Cada combinação `seed × fold` concluída produz resumo, histórico, previsões e checkpoint final. Na reexecução, essas combinações são detectadas e puladas.

### 7.2 Nível época

A v3.2.1 salva, após cada época:

- pesos atuais;
- melhor checkpoint;
- estado do AdamW;
- estado do scheduler;
- histórico;
- contagem de early stopping;
- estado do DataLoader;
- estados aleatórios Python, NumPy, CPU e GPU.

Se a execução for interrompida no meio de uma época, a retomada parte do início dessa mesma época.

### 7.3 Limite da migração

Uma combinação interrompida pela versão antiga não possui o checkpoint completo de retomada. Ela pode reiniciar uma única vez. As combinações já concluídas continuam preservadas.

## Parte 8 — Experimentos para responder aos revisores

O pacote experimental inclui:

1. TCCT completo em 5 folds × 3 seeds;
2. ablação amplitude-only;
3. ablação sem coatenção;
4. CNN;
5. ViT;
6. Random Forest;
7. consolidação de média/desvio-padrão;
8. gating modal;
9. fine-tuning espacial seguro.

A limitação de sítio único não é eliminada por esses experimentos e deve permanecer explícita no artigo.

## Parte 9 — Holdout espacial 70/20/10

### 9.1 Finalidade

Este experimento adicional usa todas as 10.500 amostras distribuídas em três faixas espaciais contíguas:

- aproximadamente 70% para treino;
- aproximadamente 20% para validação/early stopping;
- aproximadamente 10% para teste final intocado.

### 9.2 Paciência maior

A configuração opcional amplia o máximo e a paciência das Fases A e B para verificar se o TCCT continua aprendendo.

### 9.3 Limite metodológico

Como todas as amostras são usadas, não há faixa-buffer entre as três partições. Por isso, o experimento é complementar. O resultado publicável principal continua sendo a validação cruzada espacial multiseed.

Depois de avaliar o teste final, pode-se treinar um modelo de produção com treino+validação, mas esse novo modelo não deve ser reavaliado no mesmo teste para ajustar decisões.

## Parte 10 — Data augmentation

Rotações e espelhamentos permanecem desligados.

Em SAR, range e azimute possuem significado físico; rotações podem criar orientações que não ocorreram na aquisição e impor invariância inadequada à fase e às texturas. Augmentation também não cria novas amostras espacialmente independentes.

Se for estudado, deve ser um experimento separado, aplicado apenas ao treino e comparado nos mesmos folds/seeds.

## Parte 11 — Checklist antes de atualizar o artigo

- Confirmar 15 execuções do TCCT completo.
- Verificar quantas selecionaram Fase A ou Fase B.
- Consolidar RMSE, R², viés e dispersões.
- Comparar ablações e baselines.
- Atualizar o número de patches para 10.500.
- Descrever `.grille` 3D, datum vertical e coerência normalizada.
- Descrever o split como validação espacial por grupos, sem afirmar independência absoluta com buffer.
- Corrigir a interpretação da figura XAI.
- Manter a limitação de um único sítio.
- Separar resultados reproduzidos de valores apenas declarados em versões antigas do manuscrito.